# Analisis conjunto - Campeche

> La composicion de las unidades economicas de Campeche, se corresponde con el comportamiento de su actividad economica agregada?

**Division de trabajo ("por partes"):**
- Seccion 1 (consulta JOIN): **Aremy**
- Seccion 2 (grafica): **Abigail**
- Secciones 3 y 4 (conclusion y limitaciones): las dos

**Estado del 19/ago/2026:** la capa macro (BIE) esta cargada (4 series, 2022-Q1 a 2026-Q1). La capa micro (DENUE) quedo vacia porque el servicio del INEGI estaba cerrando conexiones; cuando regrese, reejecutar `python main.py` en la rama `feature/integracion` y volver a correr este notebook.

El SQL y la grafica estan escritos para funcionar en cuanto los datos existan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from core.db import consultar

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Seccion 1 · Consulta con JOIN (Aremy)

Puente: `denue_establecimiento -> dim_sector_actividad (sector_id) -> gran_division <- -> bie_indicador.gran_division -> bie_observacion (indicador_id)`.

Dos advertencias que hay que tener presentes:
1. **`Total` no aparece**: `dim_sector_actividad` solo tiene `Primarias/Secundarias/Terciarias`, asi que la serie `Total` del ITAEE no tiene contraparte micro. Queda fuera por diseno.
2. **Micro = fotografia, macro = serie de tiempo**: el conteo de establecimientos es un numero fijo que se repite en cada trimestre al hacer el JOIN.

In [ ]:
SQL_JOIN = '''
WITH micro AS (
    SELECT d.gran_division,
           COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
macro AS (
    SELECT bi.gran_division,
           o.anio,
           o.trimestre,
           o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
)
SELECT mi.gran_division,
       mi.n_establecimientos,
       ma.anio,
       ma.trimestre,
       ma.valor
FROM micro mi
JOIN macro ma ON mi.gran_division = ma.gran_division
ORDER BY mi.gran_division, ma.anio, ma.trimestre;
'''

df = consultar(SQL_JOIN)
print('Filas del JOIN:', len(df))
df.head()

In [ ]:
# Variante: conteo vs valor del ITAEE en el trimestre mas reciente.
# Evita repetir el conteo en cada trimestre y da el cuadro "aqui y ahora".
SQL_PUNTO = '''
WITH micro AS (
    SELECT d.gran_division, COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
ultimo AS (
    SELECT bi.gran_division, o.periodo
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
      AND o.periodo = (SELECT MAX(o2.periodo) FROM bie_observacion o2
                       WHERE o2.area_geografica = '04'
                         AND o2.indicador_id = o.indicador_id)
),
macro AS (
    SELECT bi.gran_division, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    JOIN ultimo u ON u.gran_division = bi.gran_division
                 AND u.periodo = o.periodo
    WHERE o.area_geografica = '04'
)
SELECT m.gran_division,
       m.n_establecimientos,
       ROUND(m.n_establecimientos * 100.0 / SUM(m.n_establecimientos) OVER (), 1) AS pct_establecimientos,
       ROUND(ma.valor, 2) AS itaee_ultimo_trimestre
FROM micro m
JOIN macro ma ON m.gran_division = ma.gran_division;
'''

df_punto = consultar(SQL_PUNTO)
print('Cuadro final (micro vs macro, ultimo trimestre):')
df_punto

## Seccion 2 · Grafica (Abigail)

Los conteos (micro) y los indices (macro) son magnitudes incomparables. La solucion elegida:
- Panel izquierdo: **participacion (%)** de los establecimientos por gran division (composicion micro).
- Panel derecho: serie del **ITAEE** (indice base 2018=100) por gran division (comportamiento macro).

Asi se leen las dos cosas sin fingir que comparten escala.

In [ ]:
# Micro: participacion por gran_division (si la capa ya esta cargada).
micro = consultar('''
    SELECT d.gran_division, COUNT(*) AS n
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
''')

# Macro: serie completa por gran_division.
macro = consultar('''
    SELECT bi.gran_division, o.anio, o.trimestre, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
''')
macro['periodo'] = macro['anio'].astype(str) + '-' + macro['trimestre'].astype(str)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

if micro.empty:
    ax1.text(0.5, 0.5, 'Sin datos micro (DENUE no cargado).\nReejecutar `python main.py`.',
             ha='center', va='center', transform=ax1.transAxes)
else:
    micro = micro.sort_values('n', ascending=False)
    ax1.bar(micro['gran_division'], micro['n'] / micro['n'].sum() * 100,
            color=['#4C72B0', '#DD8452', '#55A868'])
    ax1.set_ylabel('Participacion en establecimientos (%)')
    ax1.set_title('Composicion micro (DENUE)')

for gd in ['Primarias', 'Secundarias', 'Terciarias']:
    s = macro[macro['gran_division'] == gd].sort_values(['anio', 'trimestre'])
    ax2.plot(s['periodo'], s['valor'], marker='o', label=gd)
ax2.set_ylabel('ITAEE (base 2018 = 100)')
ax2.set_title('Actividad macro (BIE) - Campeche')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Numeros clave reproducibles para la conclusion.
if not micro.empty:
    total = micro['n'].sum()
    print(f"Establecimientos totales: {total:,}")
    print(micro.assign(pct=round(micro['n'] / total * 100, 1)).to_string(index=False))
else:
    print('Micro vacio (DENUE no cargado aun).')

print()
print('ITAEE ultimo trimestre (2026-Q1):')
print(consultar('''
    SELECT bi.gran_division, ROUND(o.valor, 2) AS valor
    FROM bie_observacion o JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04' AND o.anio = 2026 AND o.trimestre = 1
''').to_string(index=False))

## Seccion 3 · Conclusion (media cuartilla, las dos)

**Borrador** (ajustar con los numeros reales que salgan de las celdas):

Campeche tiene cerca de 48 mil negocios registrados, y casi nueve de cada diez son comercios y servicios. Pero la actividad economica medida por el INEGI cuenta otra historia: el indice de las actividades secundarias -donde vive la extraccion- cayo alrededor de 20% entre 2022 y 2026 y sigue muy por debajo de su nivel base, mientras las primarias subieron y las terciarias quedaron planas. Es decir, la mayoria de los establecimientos del estado pertenece a las actividades que menos pesan, o que mas se han estancado, en la economia agregada. Tener miles de negocios chicos no equivale a una economia de servicios: el peso real lo sigue marcando un sector con relativamente pocos establecimientos. Los datos de los que sale esta afirmacion son el conteo de establecimientos del DENUE por gran division y el ITAEE trimestral del BIE, unidos por la clasificacion SCIAN.

## Seccion 4 · Limitaciones (las dos)

**Borrador** (editar):

- El DENUE cuenta **establecimientos**, no empleo ni produccion. Una cadena con 40 sucursales aparece 40 veces.
- El ITAEE es un **indice** (base 2018=100), no un monto: no se puede sumar entre divisiones ni afirmar cuanto "pesa" cada sector en el producto.
- **Fotografia vs serie**: el DENUE es un corte en el tiempo; el ITAEE, una serie. Comparar su participacion presupone que la estructura actual de negocios es representativa de todo el periodo.
- No se puede concluir causalidad ni que "los comercios producen poco": para eso haria falta valor agregado por tamano o estrato.
- El conteo incluye todos los tamanos (de micro a grande); sin el estrato como llave no sabemos cuanto empleo absorbe cada sector.
- El `Total` del ITAEE queda fuera del JOIN porque el catalogo de sectores no tiene una categoria "Total".